# Apogee Predictor Comparison
Comparing different apogee prediction methods for airbrake control.

In [5]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Kinematic Predictor
Assumes no drag. Integrates current vertical velocity upward under gravity only.

$$h_{apogee} = h + \frac{v_z^2}{2g}$$

In [6]:
G = 9.81  # m/s²

def predict_apogee_kinematic(altitude, vz):
    """Predict apogee assuming no drag."""
    return altitude + (vz**2) / (2 * G)

### Simulate a coast phase to test predictor
Generate ground truth by numerically integrating with drag, then compare kinematic prediction at each timestep.

In [7]:
# Rocket parameters (Calisto-like)
mass       = 14.426 + 1.815   # kg, airframe + dry motor
radius     = 127 / 2000        # m, body radius
Cd_body    = 0.45              # drag coefficient (coast, no airbrakes)
rho        = 1.225             # kg/m³, air density (sea level, constant for simplicity)
A_ref      = np.pi * radius**2 # m², reference area

# Initial conditions at burnout
h0 = 1000.0   # m AGL
vz0 = 200.0   # m/s upward

# Simulate coast phase (ground truth) with drag
dt = 0.01
t, h, vz = 0.0, h0, vz0

t_hist, h_hist, vz_hist = [t], [h], [vz]

while vz > 0:
    drag = 0.5 * rho * Cd_body * A_ref * vz**2  # drag force (opposes motion)
    az = -G - (drag / mass)                      # deceleration
    vz += az * dt
    h  += vz * dt
    t  += dt
    t_hist.append(t)
    h_hist.append(h)
    vz_hist.append(vz)

true_apogee = max(h_hist)
print(f"True apogee (with drag): {true_apogee:.1f} m AGL")
print(f"Kinematic prediction at burnout: {predict_apogee_kinematic(h0, vz0):.1f} m AGL")

True apogee (with drag): 2462.6 m AGL
Kinematic prediction at burnout: 3038.7 m AGL


## 2. NASA Drag-Coast Predictor
Accounts for drag during coast assuming constant Cd, density, and mass.

Terminal velocity: $V_t = \sqrt{\frac{2mg}{\rho C_d A}}$

$$h_{apogee} = h + \frac{V_t^2}{2g} \ln\left(\frac{v_z^2 + V_t^2}{V_t^2}\right)$$

In [ ]:
def predict_apogee_nasa(altitude, vz, mass, rho, Cd, A_ref):
    """Predict apogee using NASA drag-coast equation."""
    Vt2 = (2 * mass * G) / (rho * Cd * A_ref)  # terminal velocity squared
    return altitude + (Vt2 / (2 * G)) * np.log((vz**2 + Vt2) / Vt2)

In [ ]:
# Compute both predictions at each timestep
kin_predictions  = [predict_apogee_kinematic(h, v) for h, v in zip(h_hist, vz_hist)]
nasa_predictions = [predict_apogee_nasa(h, v, mass, rho, Cd_body, A_ref) for h, v in zip(h_hist, vz_hist)]

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

axes[0].plot(t_hist, h_hist, 'C0', label='True altitude')
axes[0].axhline(true_apogee, color='C2', ls='--', lw=1.5, label=f'True apogee ({true_apogee:.0f} m)')
axes[0].set_ylabel('Altitude AGL (m)')
axes[0].legend()
axes[0].set_title('Coast Phase Simulation')

axes[1].plot(t_hist, kin_predictions,  'C1', label='Kinematic')
axes[1].plot(t_hist, nasa_predictions, 'C3', label='NASA drag-coast')
axes[1].axhline(true_apogee, color='C2', ls='--', lw=1.5, label=f'True apogee ({true_apogee:.0f} m)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Predicted apogee (m)')
axes[1].legend()
axes[1].set_title('Predictor Comparison')

plt.tight_layout()
plt.show()

kin_err  = predict_apogee_kinematic(h0, vz0) - true_apogee
nasa_err = predict_apogee_nasa(h0, vz0, mass, rho, Cd_body, A_ref) - true_apogee
print(f"Kinematic error at burnout:  {kin_err:+.1f} m ({100*kin_err/true_apogee:.1f}%)")
print(f"NASA drag-coast error:       {nasa_err:+.1f} m ({100*nasa_err/true_apogee:.1f}%)")